In [0]:
import logging
from pyspark.sql.functions import *
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("gold_layer")

CATALOG = "energy_catalog.analytics"

try:
    logger.info("Starting Gold layer pipeline")

    # ─── Step 1: Read Silver Sources ───────────────────────
    df_energy = spark.table("energy_catalog.processed.silver_energy_usage").select(
        "household_id", "region_name", "city_name",
        "meter_type", "customer_category",
        "energy_usage_kwh", "active_power_kw", "reactive_power_kvar",
        "peak_demand_kw", "offpeak_demand_kw"
    )

    df_device = spark.table("energy_catalog.processed.silver_device_metrics").select(
        "household_id", "device_category", "device_brand", "device_model",
        "maintenance_status", "installation_region",
        "device_power_kw", "efficiency_ratio", "device_temperature",
        "runtime_hours", "energy_draw_kwh"
    )

    df_weather = spark.table("energy_catalog.processed.silver_weather").select(
        col("weather_region").alias("region_name"),
        "climate_zone", "temperature_celsius", "humidity_percent"
    )

    df_grid = spark.table("energy_catalog.processed.silver_grid_load").select(
        col("grid_region").alias("region_name"),
        "grid_load_kw", "transformer_load", "line_loss_percent"
    )

    df_tariff = spark.table("energy_catalog.processed.silver_tariff_metrics").select(
        "household_id", "tariff_region", "tariff_plan_type",
        "billing_cycle", "utility_provider",
        "unit_rate", "peak_rate", "offpeak_rate",
        "fixed_charge", "subsidy_amount",
        "monthly_bill", "billing_units"
    )

    logger.info("Silver columns loaded")

    # ─── Step 2: dim_household ─────────────────────────────
    # FIX: groupBy on household_id ONLY
    # billing_cycle/utility_provider → first() not groupBy key
    # otherwise 1 household with 2 billing cycles = 2 rows
    tariff_agg = df_tariff.groupBy("household_id").agg(
        first("billing_cycle")  .alias("billing_cycle"),
        first("utility_provider").alias("utility_provider"),
        avg("monthly_bill")     .alias("avg_monthly_bill"),
        sum("billing_units")    .alias("total_units_consumed"),
        count("*")              .alias("total_records")
    )

    energy_hh = df_energy.select(
        "household_id", "meter_type", "customer_category",
        "city_name", "region_name"
    ).dropDuplicates(["household_id"])

    dim_household = energy_hh \
        .join(tariff_agg, on="household_id", how="left") \
        .withColumn("household_sk",
            row_number().over(Window.orderBy("household_id"))
        ).select(
            "household_sk", "household_id",
            "meter_type", "customer_category",
            "billing_cycle", "utility_provider",
            "city_name", "region_name",
            "total_records", "avg_monthly_bill", "total_units_consumed"
        )

    logger.info("dim_household created")

    # ─── Step 3: dim_device ────────────────────────────────
    dim_device = df_device.groupBy(
        "device_category", "device_brand", "device_model",
        "installation_region", "maintenance_status"
    ).agg(
        count("*")                .alias("device_records"),
        avg("runtime_hours")      .alias("avg_runtime_hours"),
        avg("efficiency_ratio")   .alias("avg_efficiency"),
        avg("device_power_kw")    .alias("avg_power_consumption"),
        max("device_temperature") .alias("max_device_temperature")
    ).withColumn("device_sk", row_number().over(Window.orderBy("device_model")))

    logger.info("dim_device created")

    # ─── Step 4: dim_location ──────────────────────────────
    # FIX: groupBy on region_name ONLY
    # climate_zone → first() not groupBy key
    # otherwise 1 region with 2 climate zones = 2 rows → fact fan-out
    weather_agg = df_weather.groupBy("region_name").agg(
        first("climate_zone")            .alias("climate_zone"),
        avg("temperature_celsius")       .alias("avg_temperature"),
        avg("humidity_percent")          .alias("avg_humidity")
    )

    grid_agg = df_grid.groupBy("region_name").agg(
        avg("grid_load_kw").alias("avg_grid_load"),
        max("grid_load_kw").alias("peak_grid_load")
    )

    dim_location = df_energy.select("region_name", "city_name") \
        .dropDuplicates(["region_name"]) \
        .join(weather_agg, on="region_name", how="left") \
        .join(grid_agg,    on="region_name", how="left") \
        .withColumn("location_sk",
            row_number().over(Window.orderBy("region_name"))
        ).select(
            "location_sk", "region_name", "city_name", "climate_zone",
            "avg_temperature", "avg_humidity",
            "avg_grid_load", "peak_grid_load"
        )

    logger.info("dim_location created")

    # ─── Step 5: dim_tariff ────────────────────────────────
    # FIX: groupBy on tariff_plan_type ONLY
    # tariff_region → first() not groupBy key
    # otherwise 1 plan type across 4 regions = 4 rows → fact fan-out
    dim_tariff = df_tariff.groupBy("tariff_plan_type").agg(
        first("tariff_region")  .alias("tariff_region"),
        avg("unit_rate")        .alias("avg_unit_rate"),
        avg("peak_rate")        .alias("avg_peak_rate"),
        avg("offpeak_rate")     .alias("avg_offpeak_rate"),
        avg("fixed_charge")     .alias("avg_fixed_charge"),
        avg("subsidy_amount")   .alias("avg_subsidy")
    ).withColumn("tariff_sk", row_number().over(Window.orderBy("tariff_plan_type")))

    logger.info("dim_tariff created")

    # ─── Step 6: Schema ────────────────────────────────────
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}")

    # ─── Step 7: Write Dims ────────────────────────────────
    for df, name in [
        (dim_household, "dim_household"),
        (dim_device,    "dim_device"),
        (dim_location,  "dim_location"),
        (dim_tariff,    "dim_tariff")
    ]:
        df.write.format("delta") \
            .mode("overwrite").option("overwriteSchema", "true") \
            .saveAsTable(f"{CATALOG}.{name}")
        logger.info(f"{name} written ✓")

    # ─── Step 8: Build fact_energy_consumption ─────────────
    hh_sk = broadcast(
        spark.table(f"{CATALOG}.dim_household")
             .select("household_id", "household_sk")
    )

    dev_sk = broadcast(
        spark.table(f"{CATALOG}.dim_device")
             .dropDuplicates(["device_model"])
             .select("device_model", "device_sk")
    )

    loc_sk = broadcast(
        spark.table(f"{CATALOG}.dim_location")
             .select("region_name", "location_sk")
    )

    tar_sk = broadcast(
        spark.table(f"{CATALOG}.dim_tariff")
             .select("tariff_plan_type", "tariff_sk", "avg_unit_rate")
    )

    # Device: 1 row per household
    w_dev = Window.partitionBy("household_id").orderBy("device_model")
    dm = df_device.select(
        "household_id", "device_model", "runtime_hours", "energy_draw_kwh"
    ).withColumn("rn", row_number().over(w_dev)) \
     .filter(col("rn") == 1).drop("rn") \
     .join(dev_sk, on="device_model", how="left") \
     .select("household_id", "device_sk", "runtime_hours", "energy_draw_kwh")

    # Tariff: 1 row per household — tariff_plan_type dropped after SK resolved
    w_tar = Window.partitionBy("household_id").orderBy(col("monthly_bill").desc())
    t = df_tariff.select(
        "household_id", "tariff_plan_type", "monthly_bill", "billing_units"
    ).withColumn("rn", row_number().over(w_tar)) \
     .filter(col("rn") == 1).drop("rn") \
     .join(tar_sk, on="tariff_plan_type", how="left") \
     .select("household_id", "tariff_sk", "avg_unit_rate",
             "monthly_bill", "billing_units")

    # Weather: 1 row per region
    w = df_weather.groupBy("region_name").agg(
        avg("temperature_celsius").alias("temperature_celsius"),
        avg("humidity_percent")   .alias("humidity_percent")
    )

    # Grid: 1 row per region
    g = df_grid.groupBy("region_name").agg(
        avg("grid_load_kw")      .alias("grid_load_kw"),
        avg("transformer_load")  .alias("transformer_load"),
        avg("line_loss_percent") .alias("line_loss_percent")
    )

    e = df_energy.select(
        "household_id", "region_name",
        "energy_usage_kwh", "active_power_kw", "reactive_power_kvar",
        "peak_demand_kw", "offpeak_demand_kw"
    )

    fact_energy_consumption = e \
        .join(hh_sk,  on="household_id", how="left") \
        .join(dm,     on="household_id", how="left") \
        .join(t,      on="household_id", how="left") \
        .join(loc_sk, on="region_name",  how="left") \
        .join(w,      on="region_name",  how="left") \
        .join(g,      on="region_name",  how="left") \
        .withColumn("fact_sk", monotonically_increasing_id()) \
        .withColumn("energy_cost",
            round(col("energy_usage_kwh") * col("avg_unit_rate"), 4)
        ) \
        .withColumn("power_factor",
            round(col("active_power_kw") / nullif(col("reactive_power_kvar"), lit(0)), 4)
        ) \
        .withColumn("device_efficiency",
            round(col("energy_draw_kwh") / nullif(col("runtime_hours"), lit(0)), 4)
        ) \
        .withColumn("grid_stress_ratio",
            round(col("grid_load_kw") / nullif(col("transformer_load"), lit(0)), 4)
        ) \
        .select(
            "fact_sk", "household_sk", "device_sk", "location_sk", "tariff_sk",
            "energy_usage_kwh", "active_power_kw", "reactive_power_kvar",
            "peak_demand_kw", "offpeak_demand_kw",
            "grid_load_kw", "transformer_load", "line_loss_percent",
            "temperature_celsius", "humidity_percent",
            "monthly_bill", "billing_units", "runtime_hours", "energy_draw_kwh",
            "energy_cost", "power_factor", "device_efficiency", "grid_stress_ratio"
        )

    logger.info("fact_energy_consumption dataframe built")

    # ─── Step 9: Write Fact ────────────────────────────────
    fact_energy_consumption.write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.fact_energy_consumption")
    logger.info("fact_energy_consumption written ✓")

    # ─── Step 10: Verify ───────────────────────────────────
    for tbl in ["dim_household", "dim_device", "dim_location",
                "dim_tariff", "fact_energy_consumption"]:
        cnt = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {CATALOG}.{tbl}"
        ).collect()[0]["cnt"]
        logger.info(f"  {tbl:<30} : {cnt} rows")

    logger.info("Gold layer pipeline — SUCCESSFUL ✓")

except Exception as e:
    logger.error("Gold pipeline failed")
    logger.error(str(e))
    raise

finally:
    logger.info("Gold pipeline execution completed")

In [0]:
import logging
from pyspark.sql.functions import *
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("gold_business_insights")
 
CATALOG = "energy_catalog.analytics"
 
try:
    logger.info("Starting Business Insights pipeline")
 
    # ──────────────────────────────────────────────────
    # Load Gold tables
    # ──────────────────────────────────────────────────
    fact   = spark.table(f"{CATALOG}.fact_energy_consumption")
    dim_hh = spark.table(f"{CATALOG}.dim_household")
    dim_dv = spark.table(f"{CATALOG}.dim_device")
    dim_lo = spark.table(f"{CATALOG}.dim_location")
    dim_tr = spark.table(f"{CATALOG}.dim_tariff")
 
    logger.info("Gold tables loaded for business insights")
 
    # ──────────────────────────────────────────────────
    # 1. metrics_regional_energy_summary
    #    Grain: 1 row per region
    #    Purpose: Executive dashboard — regional KPIs
    # ──────────────────────────────────────────────────
    fact_with_region = fact.join(
        dim_lo.select("location_sk", "region_name", "city_name", "climate_zone"),
        on="location_sk", how="left"
    )
 
    metrics_regional = fact_with_region.groupBy("region_name", "city_name", "climate_zone").agg(
        count("*")                          .alias("total_readings"),
        countDistinct("household_sk")       .alias("unique_households"),
        round(avg("energy_usage_kwh"), 4)   .alias("avg_energy_usage_kwh"),
        round(sum("energy_usage_kwh"), 2)   .alias("total_energy_usage_kwh"),
        round(avg("peak_demand_kw"), 4)     .alias("avg_peak_demand_kw"),
        round(max("peak_demand_kw"), 4)     .alias("max_peak_demand_kw"),
        round(avg("offpeak_demand_kw"), 4)  .alias("avg_offpeak_demand_kw"),
        round(avg("energy_cost"), 4)        .alias("avg_energy_cost"),
        round(sum("energy_cost"), 2)        .alias("total_energy_cost"),
        round(avg("monthly_bill"), 2)       .alias("avg_monthly_bill"),
        round(avg("grid_load_kw"), 4)       .alias("avg_grid_load_kw"),
        round(avg("line_loss_percent"), 4)  .alias("avg_line_loss_pct"),
        round(avg("temperature_celsius"), 2).alias("avg_temperature_c"),
        round(avg("humidity_percent"), 2)   .alias("avg_humidity_pct")
    ).withColumn("peak_to_offpeak_ratio",
        round(col("avg_peak_demand_kw") / nullif(col("avg_offpeak_demand_kw"), lit(0)), 4)
    ).withColumn("cost_per_kwh",
        round(col("total_energy_cost") / nullif(col("total_energy_usage_kwh"), lit(0)), 4)
    ).withColumn("generated_at", current_timestamp())
 
    metrics_regional.write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.metrics_regional_energy_summary")
 
    cnt = metrics_regional.count()
    logger.info(f"metrics_regional_energy_summary written ✓ — {cnt} rows")
 
    # ──────────────────────────────────────────────────
    # 2. metrics_household_consumption_tiers
    #    Grain: 1 row per household
    #    Purpose: Segment households for targeted programs
    #    Tiers based on percentile rank of avg usage
    # ──────────────────────────────────────────────────
    hh_usage = fact.groupBy("household_sk").agg(
        count("*")                             .alias("total_readings"),
        round(avg("energy_usage_kwh"), 4)      .alias("avg_usage_kwh"),
        round(sum("energy_usage_kwh"), 2)      .alias("total_usage_kwh"),
        round(avg("peak_demand_kw"), 4)        .alias("avg_peak_demand_kw"),
        round(avg("energy_cost"), 4)           .alias("avg_energy_cost"),
        round(sum("energy_cost"), 2)           .alias("total_energy_cost"),
        round(avg("monthly_bill"), 2)          .alias("avg_monthly_bill"),
        round(avg("device_efficiency"), 4)     .alias("avg_device_efficiency"),
        round(avg("power_factor"), 4)          .alias("avg_power_factor")
    )
 
    # Percentile-based tier assignment
    hh_usage = hh_usage.withColumn("usage_percentile",
        percent_rank().over(Window.orderBy("avg_usage_kwh"))
    ).withColumn("consumption_tier",
        when(col("usage_percentile") <= 0.25, "Low")
        .when(col("usage_percentile") <= 0.50, "Medium")
        .when(col("usage_percentile") <= 0.75, "High")
        .otherwise("Very High")
    ).withColumn("cost_efficiency_flag",
        when(col("avg_energy_cost") > 0,
            round(col("avg_usage_kwh") / col("avg_energy_cost"), 4)
        ).otherwise(lit(None))
    )
 
    metrics_hh_tiers = hh_usage.join(
        dim_hh.select("household_sk", "household_id", "meter_type",
                      "customer_category", "region_name", "city_name"),
        on="household_sk", how="left"
    ).withColumn("generated_at", current_timestamp()) \
     .select(
        "household_sk", "household_id",
        "meter_type", "customer_category", "region_name", "city_name",
        "total_readings", "avg_usage_kwh", "total_usage_kwh",
        "avg_peak_demand_kw", "avg_energy_cost", "total_energy_cost",
        "avg_monthly_bill", "avg_device_efficiency", "avg_power_factor",
        "usage_percentile", "consumption_tier", "cost_efficiency_flag",
        "generated_at"
    )
 
    metrics_hh_tiers.write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.metrics_household_consumption_tiers")
 
    cnt = metrics_hh_tiers.count()
    logger.info(f"metrics_household_consumption_tiers written ✓ — {cnt} rows")
 
    # ──────────────────────────────────────────────────
    # 3. metrics_device_efficiency_ranking
    #    Grain: 1 row per device_category × device_brand
    #    Purpose: Identify best/worst performing devices
    # ──────────────────────────────────────────────────
    metrics_device = dim_dv.groupBy("device_category", "device_brand").agg(
        count("*")                                  .alias("total_device_variants"),
        round(avg("avg_efficiency"), 4)             .alias("avg_efficiency_ratio"),
        round(avg("avg_runtime_hours"), 2)          .alias("avg_runtime_hours"),
        round(avg("avg_power_consumption"), 4)      .alias("avg_power_kw"),
        round(max("max_device_temperature"), 2)     .alias("peak_temperature"),
        sum("device_records")                       .alias("total_device_records"),
        round(
            avg("avg_power_consumption") /
            nullif(avg("avg_efficiency"), lit(0)), 4
        ).alias("power_per_efficiency_unit")
    )
 
    # Rank within each category
    w_eff = Window.partitionBy("device_category").orderBy(col("avg_efficiency_ratio").desc())
    metrics_device = metrics_device \
        .withColumn("efficiency_rank_in_category", row_number().over(w_eff)) \
        .withColumn("efficiency_grade",
            when(col("avg_efficiency_ratio") >= 0.80, "A")
            .when(col("avg_efficiency_ratio") >= 0.65, "B")
            .when(col("avg_efficiency_ratio") >= 0.55, "C")
            .otherwise("D")
        ) \
        .withColumn("maintenance_risk",
            when(col("peak_temperature") > 55, "HIGH")
            .when(col("peak_temperature") > 45, "MEDIUM")
            .otherwise("LOW")
        ) \
        .withColumn("generated_at", current_timestamp())
 
    metrics_device.write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.metrics_device_efficiency_ranking")
 
    cnt = metrics_device.count()
    logger.info(f"metrics_device_efficiency_ranking written ✓ — {cnt} rows")
 
    # ──────────────────────────────────────────────────
    # 4. metrics_tariff_plan_comparison
    #    Grain: 1 row per tariff_plan_type
    #    Purpose: Compare cost-effectiveness of plans
    # ──────────────────────────────────────────────────
    fact_with_tariff = fact.join(
        dim_tr.select("tariff_sk", "tariff_plan_type",
                      "avg_unit_rate", "avg_peak_rate", "avg_offpeak_rate",
                      "avg_fixed_charge", "avg_subsidy"),
        on="tariff_sk", how="left"
    )
 
    metrics_tariff = fact_with_tariff.groupBy(
        "tariff_plan_type", "avg_unit_rate", "avg_peak_rate",
        "avg_offpeak_rate", "avg_fixed_charge", "avg_subsidy"
    ).agg(
        countDistinct("household_sk")            .alias("enrolled_households"),
        count("*")                               .alias("total_readings"),
        round(avg("energy_usage_kwh"), 4)        .alias("avg_usage_kwh"),
        round(sum("energy_usage_kwh"), 2)        .alias("total_usage_kwh"),
        round(avg("energy_cost"), 4)             .alias("avg_energy_cost"),
        round(avg("monthly_bill"), 2)            .alias("avg_monthly_bill"),
        round(sum("monthly_bill"), 2)            .alias("total_revenue"),
        round(avg("billing_units"), 2)           .alias("avg_billing_units")
    ).withColumn("revenue_per_household",
        round(col("total_revenue") / nullif(col("enrolled_households"), lit(0)), 2)
    ).withColumn("cost_per_kwh",
        round(col("avg_energy_cost") / nullif(col("avg_usage_kwh"), lit(0)), 4)
    ).withColumn("peak_offpeak_spread",
        round(col("avg_peak_rate") - col("avg_offpeak_rate"), 4)
    ).withColumn("subsidy_coverage_pct",
        round(col("avg_subsidy") / nullif(col("avg_monthly_bill"), lit(0)) * 100, 2)
    ).withColumn("generated_at", current_timestamp())
 
    metrics_tariff.write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.metrics_tariff_plan_comparison")
 
    cnt = metrics_tariff.count()
    logger.info(f"metrics_tariff_plan_comparison written ✓ — {cnt} rows")
 
    # ──────────────────────────────────────────────────
    # 5. metrics_grid_health_overview
    #    Grain: 1 row per region
    #    Purpose: Grid infrastructure health monitoring
    # ──────────────────────────────────────────────────
    metrics_grid = fact_with_region.groupBy("region_name").agg(
        count("*")                                .alias("total_readings"),
        round(avg("grid_load_kw"), 4)             .alias("avg_grid_load_kw"),
        round(max("grid_load_kw"), 4)             .alias("peak_grid_load_kw"),
        round(avg("transformer_load"), 4)         .alias("avg_transformer_load"),
        round(avg("line_loss_percent"), 4)        .alias("avg_line_loss_pct"),
        round(max("line_loss_percent"), 4)        .alias("max_line_loss_pct"),
        round(avg("grid_stress_ratio"), 4)        .alias("avg_grid_stress_ratio"),
        round(max("grid_stress_ratio"), 4)        .alias("max_grid_stress_ratio"),
        round(avg("energy_usage_kwh"), 4)         .alias("avg_demand_kwh"),
        round(sum("energy_usage_kwh"), 2)         .alias("total_demand_kwh")
    ).withColumn("grid_health_status",
        when(col("avg_grid_stress_ratio") > 1.5, "CRITICAL")
        .when(col("avg_grid_stress_ratio") > 1.0, "STRESSED")
        .when(col("avg_grid_stress_ratio") > 0.7, "MODERATE")
        .otherwise("HEALTHY")
    ).withColumn("line_loss_flag",
        when(col("avg_line_loss_pct") > 10, "HIGH_LOSS")
        .when(col("avg_line_loss_pct") > 5,  "MODERATE_LOSS")
        .otherwise("ACCEPTABLE")
    ).withColumn("capacity_headroom_pct",
        round(
            (lit(1) - col("avg_grid_stress_ratio")) * 100, 2
        )
    ).withColumn("generated_at", current_timestamp())
 
    metrics_grid.write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.metrics_grid_health_overview")
 
    cnt = metrics_grid.count()
    logger.info(f"metrics_grid_health_overview written ✓ — {cnt} rows")
 
    # ──────────────────────────────────────────────────
    # 6. metrics_weather_energy_correlation
    #    Grain: 1 row per region × climate_zone
    #    Purpose: Quantify weather impact on consumption
    # ──────────────────────────────────────────────────
    metrics_weather = fact_with_region.groupBy("region_name", "climate_zone").agg(
        count("*")                                .alias("total_readings"),
        round(avg("temperature_celsius"), 2)      .alias("avg_temperature_c"),
        round(avg("humidity_percent"), 2)         .alias("avg_humidity_pct"),
        round(avg("energy_usage_kwh"), 4)         .alias("avg_usage_kwh"),
        round(sum("energy_usage_kwh"), 2)         .alias("total_usage_kwh"),
        round(avg("peak_demand_kw"), 4)           .alias("avg_peak_demand_kw"),
        round(avg("energy_cost"), 4)              .alias("avg_energy_cost"),
        round(stddev("energy_usage_kwh"), 4)      .alias("usage_stddev"),
        round(avg("grid_load_kw"), 4)             .alias("avg_grid_load_kw")
    ).withColumn("temp_usage_index",
        round(col("avg_usage_kwh") / nullif(col("avg_temperature_c"), lit(0)), 4)
    ).withColumn("humidity_usage_index",
        round(col("avg_usage_kwh") / nullif(col("avg_humidity_pct"), lit(0)), 4)
    ).withColumn("usage_volatility",
        round(col("usage_stddev") / nullif(col("avg_usage_kwh"), lit(0)), 4)
    ).withColumn("weather_impact_category",
        when(col("avg_temperature_c") > 35, "EXTREME_HEAT")
        .when(col("avg_temperature_c") > 25, "WARM")
        .when(col("avg_temperature_c") > 15, "MILD")
        .when(col("avg_temperature_c") > 5,  "COOL")
        .otherwise("COLD")
    ).withColumn("generated_at", current_timestamp())
 
    metrics_weather.write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.metrics_weather_energy_correlation")
 
    cnt = metrics_weather.count()
    logger.info(f"metrics_weather_energy_correlation written ✓ — {cnt} rows")
 
    # ──────────────────────────────────────────────────
    # Verify all metrics tables
    # ──────────────────────────────────────────────────
    logger.info("─── Business Insights Table Counts ───")
    for tbl in [
        "metrics_regional_energy_summary",
        "metrics_household_consumption_tiers",
        "metrics_device_efficiency_ranking",
        "metrics_tariff_plan_comparison",
        "metrics_grid_health_overview",
        "metrics_weather_energy_correlation"
    ]:
        cnt = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {CATALOG}.{tbl}"
        ).collect()[0]["cnt"]
        logger.info(f"  {tbl:<45} : {cnt} rows")
 
    logger.info("Business Insights pipeline — SUCCESSFUL ✓")
 
except Exception as e:
    logger.error("Business Insights pipeline FAILED")
    logger.error(str(e))
    raise
 
finally:
    logger.info("Business Insights pipeline execution completed")